In [12]:
import os
import pandas as pd
from pymongo import MongoClient
from pymongo.errors import BulkWriteError
import time
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

MONGODB_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGODB_URI)
db = client['covid']

start_time = time.time()
total_inserted = 0
total_duplicates = 0
chunk_number = 0
CHUNK_SIZE = 50000


root = Path.cwd().resolve().parent

def get_zip_paths():
  paths = []

  for file in os.listdir(root / "data"):
    if file.endswith(".zip"):
      paths.append(file)
  return sorted(paths)

zip_files = get_zip_paths()

collection_map = {
    "2020": "raw_2020_data",
    "2021": "raw_2021_data",
    "2022": "raw_2022_data",
    "2023": "raw_2023_data",
    "2024": "raw_2024_data",
    "2025": "raw_2025_data",
    "2026": "raw_2026_data",
}



for zip_file in zip_files:
  detected_year = None

  for year in collection_map:
    if year in zip_file:
      detected_year = year
      break

  if not detected_year:
    print(f"Skipping file (not detected): {zip_file}")
    continue

  collection_name = collection_map[detected_year]

  print(f"\nProcessing: {zip_file} | Collection: {collection_name}")

  collection = db[collection_name]
  collection.create_index("ENTIDAD_RES")

  
  csv_iter = pd.read_csv(
      root / "data" / zip_file,
      dtype=str,
      chunksize=CHUNK_SIZE,
      keep_default_na=False,
      compression='zip'
  )


  for chunk in csv_iter:
    docs = []
    chunk_number += 1
    for row in chunk.to_dict(orient='records'):
      row["_id"] = row["ID_REGISTRO"]
      docs.append(row)

    if not docs:
      continue

    try:
      result = collection.insert_many(docs, ordered=False)
      inserted = len(result.inserted_ids)
      total_inserted += inserted

      print(
       f"[Chunk{chunk_number}]"
       f"Inserted {inserted} documents."
       f"Total: {total_inserted}"
      )

    except BulkWriteError as error:
      write_errors = error.details.get("writeErrors", [])
      duplicates = 0

      for err in write_errors:
        if err.get("code") == 11000:
          duplicates += 1

      inserted = error.details.get("nInserted", 0)
      total_inserted += inserted
      total_duplicates += duplicates

      print(
        f"[Chunk {chunk_number}] Inserted {inserted} documents. "
        f"Total: {total_inserted} | Duplicates: {duplicates}"
      )


elapsed = round(time.time() - start_time, 2)

print("RAW Ingestion Completed")

print(f"Total inserted: {total_inserted}")
print(f"Total duplicates: {total_duplicates}")
print(f"Elapsed time: {elapsed} seconds")



Processing: DATA_RAW_2020.zip | Collection: raw_2020_data
[Chunk1]Inserted 50000 documents.Total: 50000
[Chunk2]Inserted 50000 documents.Total: 100000
[Chunk3]Inserted 50000 documents.Total: 150000
[Chunk4]Inserted 50000 documents.Total: 200000
[Chunk5]Inserted 50000 documents.Total: 250000
[Chunk6]Inserted 50000 documents.Total: 300000
[Chunk7]Inserted 50000 documents.Total: 350000
[Chunk8]Inserted 50000 documents.Total: 400000
[Chunk9]Inserted 50000 documents.Total: 450000
[Chunk10]Inserted 50000 documents.Total: 500000
[Chunk11]Inserted 50000 documents.Total: 550000
[Chunk12]Inserted 50000 documents.Total: 600000
[Chunk13]Inserted 50000 documents.Total: 650000
[Chunk14]Inserted 50000 documents.Total: 700000
[Chunk15]Inserted 50000 documents.Total: 750000
[Chunk16]Inserted 50000 documents.Total: 800000
[Chunk17]Inserted 50000 documents.Total: 850000
[Chunk18]Inserted 50000 documents.Total: 900000
[Chunk19]Inserted 50000 documents.Total: 950000
[Chunk20]Inserted 50000 documents.Total

In [30]:
import zipfile
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
names = [f for f in os.listdir(DATA_DIR) if f.endswith('.zip')]
zip_files = [DATA_DIR / name for name in names]

for zip_file in zip_files:
    try:
        with zipfile.ZipFile(zip_file, 'r') as zf:
            print(f"'{zip_file}' is a valid zip file. It contains: {zf.namelist()}")
    except zipfile.BadZipFile:
        print(f"'{zip_file}' is NOT a valid zip file and might be corrupted. Please re-download it.")
    except FileNotFoundError:
        print(f"'{zip_file}' not found.")


'/home/netfoor/Documents/university/KDD/covid19/data/DATA_RAW_2023.zip' is a valid zip file. It contains: ['COVID19MEXICO.csv']
'/home/netfoor/Documents/university/KDD/covid19/data/DATA_RAW_2024.zip' is a valid zip file. It contains: ['COVID19MEXICO.csv']
'/home/netfoor/Documents/university/KDD/covid19/data/DATA_RAW_2026.zip' is a valid zip file. It contains: ['COVID19MEXICO.csv']
'/home/netfoor/Documents/university/KDD/covid19/data/DATA_RAW_2020.zip' is a valid zip file. It contains: ['COVID19MEXICO2020.csv']
'/home/netfoor/Documents/university/KDD/covid19/data/DATA_RAW_2025.zip' is a valid zip file. It contains: ['COVID19MEXICO.csv']
'/home/netfoor/Documents/university/KDD/covid19/data/DATA_RAW_2021.zip' is a valid zip file. It contains: ['COVID19MEXICO2021.csv']
'/home/netfoor/Documents/university/KDD/covid19/data/DATA_RAW_2022.zip' is a valid zip file. It contains: ['COVID19MEXICO2022.csv']


In [19]:
print(db.list_collection_names())

print(f"Documents in raw_2020_data: {db.raw_2020_data.count_documents({})}")
print(f"\nDocuments in raw_2021_data: {db.raw_2021_data.count_documents({})}")
print(f"\nDocuments in raw_2022_data: {db.raw_2022_data.count_documents({})}")
print(f"\nDocuments in raw_2023_data: {db.raw_2023_data.count_documents({})}")
print(f"\nDocuments in raw_2024_data: {db.raw_2024_data.count_documents({})}")
print(f"\nDocuments in raw_2025_data: {db.raw_2025_data.count_documents({})}")
print(f"\nDocuments in raw_2026_data: {db.raw_2026_data.count_documents({})}")

['raw_2023_data', 'raw_2025_data', 'raw_2022_data', 'raw_2024_data', 'raw_2021_data', 'raw_2020_data', 'raw_2026_data']
Documents in raw_2020_data: 3868396

Documents in raw_2021_data: 8830345

Documents in raw_2022_data: 6451944

Documents in raw_2023_data: 1222219

Documents in raw_2024_data: 177618

Documents in raw_2025_data: 157142

Documents in raw_2026_data: 219023


In [43]:
import zipfile
import os
from pathlib import Path
import io

path = Path.cwd().parent / "data"

for file in os.listdir(path):
    try:
        with zipfile.ZipFile(path / file, 'r') as zf:
            csv_files = [f for f in zf.namelist() if f.endswith('.csv')]
            nombre_csv = csv_files[0] if csv_files else "No CSV found"

            with zf.open(nombre_csv, 'r') as csvfile:
                f_text = io.TextIOWrapper(csvfile, encoding='utf-8')
                total_lines = sum(1 for line in f_text)
                total_registros = total_lines - 1

            print(f"'{file}' is a valid zip file. It contains: {nombre_csv} with {total_registros} registros.")
    except zipfile.BadZipFile:
        print(f"'{file}' is NOT a valid zip file and might be corrupted. Please re-download it.")
    except FileNotFoundError:
        print(f"'{file}' not found.")



'DATA_RAW_2023.zip' is a valid zip file. It contains: COVID19MEXICO.csv with 1222219 registros.
'DATA_RAW_2024.zip' is a valid zip file. It contains: COVID19MEXICO.csv with 177618 registros.
'DATA_RAW_2026.zip' is a valid zip file. It contains: COVID19MEXICO.csv with 219023 registros.
'DATA_RAW_2020.zip' is a valid zip file. It contains: COVID19MEXICO2020.csv with 3868396 registros.
'DATA_RAW_2025.zip' is a valid zip file. It contains: COVID19MEXICO.csv with 157142 registros.
'DATA_RAW_2021.zip' is a valid zip file. It contains: COVID19MEXICO2021.csv with 8830345 registros.
'DATA_RAW_2022.zip' is a valid zip file. It contains: COVID19MEXICO2022.csv with 6451944 registros.


# Correction: Delete the old collection to load the correct one.

DO NOT run this code if you want to analyze the most recent data from the official government website. This analysis uses data from May 11, 2026, and the code above works perfectly with any zip file you load into /data.

In [2]:
from src.database import get_database

db = get_database()

In [3]:
print(db.list_collection_names())

['raw_2023_data', 'covid_unified_data', 'raw_2025_data', 'raw_2022_data', 'raw_2024_data', 'raw_2021_data', 'raw_2020_data', 'raw_2026_data']


In [4]:
print(db.raw_2026_data.count_documents({}))

219023


In [7]:
db.raw_2026_data.drop()
print(sorted(db.list_collection_names()))

['covid_unified_data', 'raw_2020_data', 'raw_2021_data', 'raw_2022_data', 'raw_2023_data', 'raw_2024_data', 'raw_2025_data']


In [11]:
import os
import pandas as pd
from pymongo import MongoClient
from pymongo.errors import BulkWriteError
import time
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

MONGODB_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGODB_URI)
db = client['covid']

start_time = time.time()
total_inserted = 0
total_duplicates = 0
chunk_number = 0
CHUNK_SIZE = 50000


root = Path.cwd().resolve().parent

def get_zip_paths():
  paths = []

  for file in os.listdir(root / "data"):
    if file == 'DATA_RAW_2026.zip':
      paths.append(file)
  return sorted(paths)

zip_files = get_zip_paths()

collection_map = {
    "2026": "raw_2026_data"
}



for zip_file in zip_files:
  detected_year = None

  for year in collection_map:
    if year in zip_file:
      detected_year = year
      break

  if not detected_year:
    print(f"Skipping file (not detected): {zip_file}")
    continue

  collection_name = collection_map[detected_year]

  print(f"\nProcessing: {zip_file} | Collection: {collection_name}")

  collection = db[collection_name]
  collection.create_index("ENTIDAD_RES")

  
  csv_iter = pd.read_csv(
      root / "data" / zip_file,
      dtype=str,
      chunksize=CHUNK_SIZE,
      keep_default_na=False,
      compression='zip'
  )


  for chunk in csv_iter:
    docs = []
    chunk_number += 1
    for row in chunk.to_dict(orient='records'):
      row["_id"] = row["ID_REGISTRO"]
      docs.append(row)

    if not docs:
      continue

    try:
      result = collection.insert_many(docs, ordered=False)
      inserted = len(result.inserted_ids)
      total_inserted += inserted

      print(
       f"[Chunk{chunk_number}]"
       f"Inserted {inserted} documents."
       f"Total: {total_inserted}"
      )

    except BulkWriteError as error:
      write_errors = error.details.get("writeErrors", [])
      duplicates = 0

      for err in write_errors:
        if err.get("code") == 11000:
          duplicates += 1

      inserted = error.details.get("nInserted", 0)
      total_inserted += inserted
      total_duplicates += duplicates

      print(
        f"[Chunk {chunk_number}] Inserted {inserted} documents. "
        f"Total: {total_inserted} | Duplicates: {duplicates}"
      )


elapsed = round(time.time() - start_time, 2)

print("RAW Ingestion Completed")

print(f"Total inserted: {total_inserted}")
print(f"Total duplicates: {total_duplicates}")
print(f"Elapsed time: {elapsed} seconds")



Processing: DATA_RAW_2026.zip | Collection: raw_2026_data
[Chunk1]Inserted 50000 documents.Total: 50000
[Chunk2]Inserted 50000 documents.Total: 100000
[Chunk3]Inserted 50000 documents.Total: 150000
[Chunk4]Inserted 50000 documents.Total: 200000
[Chunk5]Inserted 17324 documents.Total: 217324
RAW Ingestion Completed
Total inserted: 217324
Total duplicates: 0
Elapsed time: 6.71 seconds


In [12]:
print(db.list_collection_names())

['raw_2023_data', 'covid_unified_data', 'raw_2026_data', 'raw_2025_data', 'raw_2022_data', 'raw_2024_data', 'raw_2021_data', 'raw_2020_data']


In [16]:
raw_count = 0 
for collection_name in sorted(db.list_collection_names()):
    count = db[collection_name].count_documents({})
    if collection_name.startswith("raw_"):
        raw_count += count
    print(f"Documents in {collection_name}: {count}")

Documents in covid_unified_data: 20769765
Documents in raw_2020_data: 3868396
Documents in raw_2021_data: 8830345
Documents in raw_2022_data: 6451944
Documents in raw_2023_data: 1222219
Documents in raw_2024_data: 177618
Documents in raw_2025_data: 157142
Documents in raw_2026_data: 217324


In [17]:
print(raw_count)

20924988


In [19]:
pipeline = [
    {
        "$group": {
            "_id": "$YEAR_ORIGIN",       
            "total": {"$sum": 1},
        }
    }
]

resultado = list(db.covid_unified_data.aggregate(pipeline))
print("Total registros por YEAR_ORIGIN:")
for doc in resultado:
    print(f"YEAR_ORIGIN: {doc['_id']}, Total: {doc['total']}")

Total registros por YEAR_ORIGIN:
YEAR_ORIGIN: 2022, Total: 6451944
YEAR_ORIGIN: 2025, Total: 224
YEAR_ORIGIN: 2020, Total: 3868395
YEAR_ORIGIN: 2021, Total: 8830342
YEAR_ORIGIN: 2024, Total: 177618
YEAR_ORIGIN: 2023, Total: 1222219
YEAR_ORIGIN: 2026, Total: 219023


In [44]:
result = db.covid_unified_data.delete_many({"YEAR_ORIGIN": 2026})

In [45]:
pipeline = [
    {
        "$group": {
            "_id": "$YEAR_ORIGIN",       
            "total": {"$sum": 1},
        }
    }
]

resultado = list(db.covid_unified_data.aggregate(pipeline))
print("Updated unified data counts:")
for doc in resultado:
    print(f"YEAR_ORIGIN: {doc['_id']}, Total: {doc['total']}")


Updated unified data counts:
YEAR_ORIGIN: 2022, Total: 6451944
YEAR_ORIGIN: 2025, Total: 224
YEAR_ORIGIN: 2023, Total: 1222219
YEAR_ORIGIN: 2024, Total: 177618
YEAR_ORIGIN: 2020, Total: 3868395
YEAR_ORIGIN: 2021, Total: 8830342


In [54]:
print(sorted(db.list_collection_names()))

['covid_unified_data', 'raw_2020_data', 'raw_2021_data', 'raw_2022_data', 'raw_2023_data', 'raw_2024_data', 'raw_2025_data', 'raw_2026_data']


In [56]:
import re
from pymongo import UpdateOne
from src.database import get_database

db = get_database()

def standardize_document(doc, year_collection):
    doc_estandarizado = {}
    campos_inconsistentes = [
        'RESULTADO_LAB', 'CLASIFICACION_FINAL', 
        'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION', 
        'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU'
    ]
    for clave, valor in doc.items():
        if clave not in campos_inconsistentes and clave != '_id':
            doc_estandarizado[clave] = valor
            
    doc_estandarizado['YEAR_ORIGIN'] = year_collection
    
    if year_collection < 2024:
        doc_estandarizado['RESULTADO_PCR'] = doc.get('RESULTADO_LAB')
        doc_estandarizado['RESULTADO_PCR_COINFECCION'] = '997' 
        doc_estandarizado['CLASIFICACION_FINAL_COVID'] = doc.get('CLASIFICACION_FINAL')
        doc_estandarizado['CLASIFICACION_FINAL_FLU'] = '6'     
    else:
        doc_estandarizado['RESULTADO_PCR'] = doc.get('RESULTADO_PCR')
        doc_estandarizado['RESULTADO_PCR_COINFECCION'] = doc.get('RESULTADO_PCR_COINFECCION')
        doc_estandarizado['CLASIFICACION_FINAL_COVID'] = doc.get('CLASIFICACION_FINAL_COVID')
        doc_estandarizado['CLASIFICACION_FINAL_FLU'] = doc.get('CLASIFICACION_FINAL_FLU')
        
    return doc_estandarizado


def procesar_historico_a_master(chunk=50000):
    coleccion_destino = db["covid_unified_data"]
    colecciones_crudas = ['raw_2026_data']
    
    for nombre_coll in colecciones_crudas:
        if not nombre_coll.startswith("raw_") or not nombre_coll.endswith("_data"):
            continue
            
        year = int(re.search(r'\d{4}', nombre_coll).group())
        print(f"\n Iniciando migración controlada de: {nombre_coll}...")
        
        cursor = db[nombre_coll].find().batch_size(chunk)
        
        operaciones_lote = []
        contador_procesados = 0
        
        for doc in cursor:
            doc_limpio = standardize_document(doc, year)
            id_registro = doc_limpio.get("ID_REGISTRO")
            
            operacion = UpdateOne(
                {"ID_REGISTRO": id_registro},
                {"$set": doc_limpio},
                upsert=True
            )
            operaciones_lote.append(operacion)
            
            if len(operaciones_lote) == chunk:
                resultado = coleccion_destino.bulk_write(operaciones_lote, ordered=False)
                contador_procesados += len(operaciones_lote)
                
                print(f"Procesados: {contador_procesados} | Nuevos creados: {resultado.upserted_count} | Actualizados: {resultado.modified_count}")
                operaciones_lote = []
                
        if operaciones_lote:
            resultado = coleccion_destino.bulk_write(operaciones_lote, ordered=False)
            contador_procesados += len(operaciones_lote)
            print(f"Procesados: {contador_procesados} | Nuevos creados: {resultado.upserted_count} | Actualizados: {resultado.modified_count}")

        print(f"¡Finalizado con éxito! {nombre_coll} completamente integrado.")

if __name__ == "__main__":

    indices_existentes = db["covid_unified_data"].index_information()
    
    if "ID_REGISTRO_1" not in indices_existentes:
        print("Creando índice único para ID_REGISTRO (esto puede tardar unos minutos)...")
        db["covid_unified_data"].create_index("ID_REGISTRO", unique=True)
        print("¡Índice creado con éxito!")
    else:
        print("El índice para ID_REGISTRO ya existe y está activo. Continuando...")
    
    
    procesar_historico_a_master()


El índice para ID_REGISTRO ya existe y está activo. Continuando...

 Iniciando migración controlada de: raw_2026_data...
Procesados: 50000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 100000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 150000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 200000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 217324 | Nuevos creados: 17324 | Actualizados: 0
¡Finalizado con éxito! raw_2026_data completamente integrado.


In [57]:
pipeline = [
    {
        "$group": {
            "_id": "$YEAR_ORIGIN",       
            "total": {"$sum": 1},
        }
    }
]

resultado = list(db.covid_unified_data.aggregate(pipeline))
print("Updated unified data counts:")
for doc in resultado:
    print(f"YEAR_ORIGIN: {doc['_id']}, Total: {doc['total']}")


Updated unified data counts:
YEAR_ORIGIN: 2022, Total: 6451944
YEAR_ORIGIN: 2025, Total: 224
YEAR_ORIGIN: 2020, Total: 3868395
YEAR_ORIGIN: 2021, Total: 8830342
YEAR_ORIGIN: 2024, Total: 177618
YEAR_ORIGIN: 2023, Total: 1222219
YEAR_ORIGIN: 2026, Total: 217324


In [59]:
unified_count = 0 
for collection_name in sorted(db.list_collection_names()):
    count = db[collection_name].count_documents({})
    if collection_name.startswith("covid_unified_"):
        unified_count += count


print(unified_count)

20768066
